# Six Degrees of MoMA: 02 Baselines

This notebook evaluates the non-learned baselines and Node2Vec on the test window of each
cutoff. They show whether graph learning beats popularity and simple network proximity.

- **popularity / recency_popularity**: prior MoMA exhibitions; the recency variant weights each
  by exp(−years before the cutoff / τ).
- **curator_collab**: sum over the target's curators of prior exhibitions shared with the artist.
- **common_neighbors / adamic_adar**: summed over the target's curators, on the projected
  person–person co-exhibition graph.
- **node2vec**: cosine similarity between the artist and the mean embedding of the target's
  curators.

Ties in each heuristic are broken by popularity, so exhibitions without a recorded curator fall
back to popularity. Outputs go to `results/baselines/`.

In [ ]:
# Installs anything missing. Colab already has torch, pandas, scikit-learn and matplotlib.
import importlib.util, subprocess, sys

REQUIRED = {"torch": "torch", "torch_geometric": "torch_geometric", "pyarrow": "pyarrow",
            "sentence_transformers": "sentence-transformers", "scipy": "scipy",
            "sklearn": "scikit-learn", "matplotlib": "matplotlib"}
missing = [pkg for mod, pkg in REQUIRED.items() if importlib.util.find_spec(mod) is None]
if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])

In [ ]:
from pathlib import Path

# On Colab, set USE_DRIVE = True to keep data and results across sessions.
USE_DRIVE = False
if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    ROOT = Path("/content/drive/MyDrive/moma-project")
else:
    ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

RAW_DIR = ROOT / "data" / "raw"
PROCESSED_DIR = ROOT / "data" / "processed"
RESULTS_DIR = ROOT / "results"
for d in [RAW_DIR, PROCESSED_DIR, RESULTS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

CFG = {
    # Raw ExhibitionRole values mapped onto node roles; everything else is "other".
    # Institutions listed with a curator role are treated as "other" (organizers).
    "roles": {"artist": ["Artist"], "curator": ["Curator"]},
    "features": {"title_encoder": "minilm",   # minilm | tfidf
                 "title_dim": 32, "top_nationalities": 12},
    "split": {
        "cutoffs": [1950, 1960, 1970, 1980],
        "val_years": 5,       # val targets:  [T - val_years, T)
        "test_years": 10,     # test targets: [T, T + test_years)
        "train_window": 5,    # rolling training targets: [t, t + train_window)
        "train_start": 1932,  # earliest pseudo-cutoff for training snapshots
    },
    "eval": {"ks": [10, 20, 50], "primary_metric": "map@20"},
    "baselines": {
        "recency_tau": 10.0,
        "node2vec": {"dim": 64, "walk_length": 20, "walks_per_node": 10, "window": 4,
                     "p": 1.0, "q": 1.0, "epochs": 2, "num_neg": 5},
    },
    "model": {
        "name": "rgcn",           # rgcn | heteroconv | hgt | sage | gat
        "curator_mode": "full",   # full | none | shuffled
        "hidden": 64, "layers": 2, "dropout": 0.2, "lr": 0.005, "weight_decay": 1e-4,
        "epochs": 300, "eval_every": 5, "patience": 10,   # patience in evaluation rounds
        "loss": "bpr",            # bpr | bce
        "num_neg": 20,
        "max_group": None,        # subsample group shows above this size (homogeneous graph)
    },
    "seeds": [0, 1, 2, 3, 4],
}

## Core: data loading and cleaning

The CSV has one row per (exhibition, constituent, role) and is cp1252-encoded.

- **Exhibitions** are keyed on (number, title, begin date). ExhibitionID is missing for about 430 rows, and
  the number "No#" covers about 50 different exhibitions. `exh_id` is the chronological index.
- **Constituents** without a ConstituentID are keyed by display name.
- **Roles**: `artist`, `curator` (an individual with `ExhibitionRole == "Curator"`) or `other`.

In [ ]:
import copy, json, time, urllib.request
from dataclasses import dataclass, replace

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

BASE_URL = "https://raw.githubusercontent.com/MuseumofModernArt/exhibitions/master/"
EXHIBITIONS_FILE = "MoMAExhibitions1929to1989.csv"
TABLES = ["exhibitions", "constituents", "participation"]


def download(raw_dir, force=False):
    dest = raw_dir / EXHIBITIONS_FILE
    if force or not dest.exists():
        print(f"downloading {EXHIBITIONS_FILE}")
        urllib.request.urlretrieve(BASE_URL + EXHIBITIONS_FILE, dest)
    return dest


def load_raw(raw_dir):
    return pd.read_csv(raw_dir / EXHIBITIONS_FILE, encoding="cp1252", dtype=str)


def build_exhibitions(raw):
    ex = raw.drop_duplicates("exh_key")[[
        "exh_key", "ExhibitionID", "ExhibitionNumber", "ExhibitionTitle",
        "ExhibitionBeginDate", "ExhibitionEndDate", "ExhibitionSortOrder", "ExhibitionURL",
    ]].copy()
    ex["begin"] = pd.to_datetime(ex["ExhibitionBeginDate"], format="%m/%d/%Y")
    ex["end"] = pd.to_datetime(ex["ExhibitionEndDate"], format="%m/%d/%Y")
    ex["sort_order"] = pd.to_numeric(ex["ExhibitionSortOrder"], errors="coerce")
    ex = ex.sort_values(["begin", "sort_order", "exh_key"]).reset_index(drop=True)
    ex["exh_id"] = np.arange(len(ex))
    ex["year"] = ex["begin"].dt.year + (ex["begin"].dt.day_of_year - 1) / 365.25
    duration = (ex["end"] - ex["begin"]).dt.days
    ex["duration_days"] = duration.where(duration >= 0)
    return ex.rename(columns={
        "ExhibitionID": "source_id", "ExhibitionNumber": "number",
        "ExhibitionTitle": "title", "ExhibitionURL": "url",
    })[["exh_id", "exh_key", "source_id", "number", "title", "begin", "end", "year",
        "duration_days", "url"]]


def build_constituents(rows):
    def first_valid(s):
        s = s.dropna()
        return s.iloc[0] if len(s) else None

    cols = {
        "DisplayName": "name", "ConstituentType": "type", "Nationality": "nationality",
        "Gender": "gender", "ConstituentBeginDate": "birth_year",
        "ConstituentEndDate": "death_year", "WikidataID": "wikidata", "ULANID": "ulan",
    }
    con = rows.groupby("cid")[list(cols)].agg(first_valid).rename(columns=cols).reset_index()
    con["gender"] = con["gender"].str.lower().fillna("unknown")
    con["nationality"] = con["nationality"].fillna("unknown")
    con["type"] = con["type"].fillna("unknown")
    for col in ["birth_year", "death_year"]:
        year = pd.to_numeric(con[col], errors="coerce")
        con[col] = year.where(year > 0)
    return con


def build_participation(rows, roles):
    raw_role = rows["ExhibitionRole"]
    role = pd.Series("other", index=rows.index)
    role[raw_role.isin(roles["artist"])] = "artist"
    is_curator = raw_role.isin(roles["curator"]) & (rows["ConstituentType"] != "Institution")
    role[is_curator] = "curator"
    part = pd.DataFrame({
        "exh_id": rows["exh_id"].values,
        "cid": rows["cid"].values,
        "role": role.values,
        "raw_role": raw_role.values,
        "press_role": rows["ExhibitionRoleinPressRelease"].values,
    })
    return part.drop_duplicates(["exh_id", "cid", "role"]).reset_index(drop=True)


def clean(raw, roles):
    raw = raw.copy()
    raw["exh_key"] = (raw["ExhibitionNumber"].fillna("") + "|" + raw["ExhibitionTitle"].fillna("")
                      + "|" + raw["ExhibitionBeginDate"].fillna(""))
    exhibitions = build_exhibitions(raw)
    raw["exh_id"] = raw["exh_key"].map(exhibitions.set_index("exh_key")["exh_id"])

    # Rows with no role and no constituent only record that an exhibition happened.
    rows = raw[raw["ExhibitionRole"].notna() & raw["DisplayName"].notna()].copy()
    rows["cid"] = ("moma:" + rows["ConstituentID"]).fillna("name:" + rows["DisplayName"].str.strip())
    constituents = build_constituents(rows)
    participation = build_participation(rows, roles)

    has_curator = participation.loc[participation["role"] == "curator", "exh_id"].unique()
    exhibitions["has_curator"] = exhibitions["exh_id"].isin(has_curator)
    return {"exhibitions": exhibitions, "constituents": constituents,
            "participation": participation}


def load_tables(force=False):
    """Cleaned tables from data/processed, rebuilding them from the raw CSV if needed."""
    if not force and all((PROCESSED_DIR / f"{n}.parquet").exists() for n in TABLES):
        return {n: pd.read_parquet(PROCESSED_DIR / f"{n}.parquet") for n in TABLES}
    download(RAW_DIR)
    tables = clean(load_raw(RAW_DIR), CFG["roles"])
    for name, df in tables.items():
        df.to_parquet(PROCESSED_DIR / f"{name}.parquet", index=False)
    return tables

### Integer-indexed dataset

Artists and curators have separate node index spaces. A person who both exhibited and curated
appears once in each; both keep `cid` so they can be merged back into one person when needed.

`with_shuffled_curators` is the shuffle ablation. It permutes curator→exhibition assignments
within each decade, so every curator keeps their number of exhibitions and every exhibition
keeps its number of curators.

In [ ]:
@dataclass
class MomaData:
    exhibitions: pd.DataFrame    # indexed by exh_id (chronological)
    artists: pd.DataFrame        # indexed by artist node id
    curators: pd.DataFrame       # indexed by curator node id
    artist_edges: pd.DataFrame   # columns exh_id, artist
    curator_edges: pd.DataFrame  # columns exh_id, curator

    @classmethod
    def from_tables(cls, tables):
        ex = tables["exhibitions"].set_index("exh_id", drop=False).sort_index()
        con = tables["constituents"].set_index("cid")
        part = tables["participation"]

        def role_nodes(role):
            edges = part[part["role"] == role][["exh_id", "cid"]]
            cids = np.sort(edges["cid"].unique())
            nodes = con.loc[cids].reset_index()
            node_of = pd.Series(np.arange(len(cids)), index=cids)
            edges = pd.DataFrame({"exh_id": edges["exh_id"].to_numpy(),
                                  role: node_of.loc[edges["cid"]].to_numpy()})
            return nodes, edges.sort_values(["exh_id", role]).reset_index(drop=True)

        artists, artist_edges = role_nodes("artist")
        curators, curator_edges = role_nodes("curator")
        return cls(ex, artists, curators, artist_edges, curator_edges)

    @property
    def num_artists(self):
        return len(self.artists)

    @property
    def num_curators(self):
        return len(self.curators)

    @property
    def num_exhibitions(self):
        return len(self.exhibitions)

    def with_shuffled_curators(self, seed):
        rng = np.random.default_rng(seed)
        edges = self.curator_edges.copy()
        decade = (self.exhibitions.loc[edges["exh_id"], "begin"].dt.year // 10).to_numpy()
        exh = edges["exh_id"].to_numpy().copy()
        for d in np.unique(decade):
            idx = np.flatnonzero(decade == d)
            exh[idx] = exh[rng.permutation(idx)]
        edges["exh_id"] = exh
        # A permutation can rarely pair a curator with the same exhibition twice.
        edges = edges.drop_duplicates().sort_values(["exh_id", "curator"]).reset_index(drop=True)
        return replace(self, curator_edges=edges)

## Core: temporal snapshots

A snapshot fixes a history cutoff and a set of target exhibitions after it. A model may see:

- every artist and curator edge of exhibitions that began before `history_end`;
- the curator edges and metadata of the target exhibitions.

It must rank the `candidates` (artists with at least one exhibition in the history) for each
target. The targets' artist edges are labels only.

For a cutoff year T:

| split | history | targets |
|---|---|---|
| train (rolling) | before t, for each t < T−5 | [t, min(t+5, T−5)) |
| validation | before T−5 | [T−5, T) |
| test | before T | [T, T+10) |

Rolling training snapshots have the same form as the test task, and the model never sees edges
from after its own targets.

In [ ]:
@dataclass
class Snapshot:
    history_end: pd.Timestamp
    targets: np.ndarray         # exh_ids, chronological
    hist_artist: np.ndarray     # [2, E] (exh_id, artist) for history exhibitions
    hist_curator: np.ndarray    # [2, E] (exh_id, curator) for history exhibitions
    target_curator: np.ndarray  # [2, E] (exh_id, curator) for target exhibitions
    candidates: np.ndarray      # artist ids with >=1 history exhibition, sorted
    positives: dict             # target exh_id -> all participating artist ids
    has_curator: np.ndarray     # per target: recorded curator in the original data

    @property
    def ref_year(self):
        ts = self.history_end
        return ts.year + (ts.day_of_year - 1) / 365.25

    @property
    def curator_edges(self):
        return np.concatenate([self.hist_curator, self.target_curator], axis=1)

    def label_matrix(self):
        """Boolean [num_targets, num_candidates]; True where the candidate exhibited."""
        labels = np.zeros((len(self.targets), len(self.candidates)), dtype=bool)
        for i, e in enumerate(self.targets):
            labels[i] = np.isin(self.candidates, self.positives[e], assume_unique=True)
        return labels


def _edges(df, col, mask):
    sub = df[mask]
    return np.stack([sub["exh_id"].to_numpy(), sub[col].to_numpy()]).astype(np.int64)


def make_snapshot(data, target_start, target_end, history_end=None):
    target_start, target_end = pd.Timestamp(target_start), pd.Timestamp(target_end)
    history_end = pd.Timestamp(history_end) if history_end is not None else target_start
    assert history_end <= target_start

    ex = data.exhibitions
    hist_ids = ex.index[ex["begin"] < history_end].to_numpy()
    targets = ex.index[(ex["begin"] >= target_start) & (ex["begin"] < target_end)].to_numpy().copy()

    ae, ce = data.artist_edges, data.curator_edges
    hist_artist = _edges(ae, "artist", ae["exh_id"].isin(hist_ids))
    hist_curator = _edges(ce, "curator", ce["exh_id"].isin(hist_ids))
    target_curator = _edges(ce, "curator", ce["exh_id"].isin(targets))

    target_artists = ae[ae["exh_id"].isin(targets)].groupby("exh_id")["artist"]
    positives = {e: np.array([], dtype=np.int64) for e in targets}
    positives.update({e: g.to_numpy() for e, g in target_artists})

    return Snapshot(
        history_end=history_end,
        targets=targets,
        hist_artist=hist_artist,
        hist_curator=hist_curator,
        target_curator=target_curator,
        candidates=np.unique(hist_artist[1]),
        positives=positives,
        has_curator=ex.loc[targets, "has_curator"].to_numpy(),
    )


def year_start(year):
    return pd.Timestamp(year=int(year), month=1, day=1)


def eval_snapshots(data, cutoff, split_cfg):
    val_start = cutoff - split_cfg["val_years"]
    val = make_snapshot(data, year_start(val_start), year_start(cutoff))
    test = make_snapshot(data, year_start(cutoff), year_start(cutoff + split_cfg["test_years"]))
    return val, test


def train_snapshots(data, cutoff, split_cfg):
    val_start = cutoff - split_cfg["val_years"]
    snaps = []
    for t in range(split_cfg["train_start"], val_start):
        end = min(t + split_cfg["train_window"], val_start)
        snap = make_snapshot(data, year_start(t), year_start(end))
        if len(snap.targets) and len(snap.candidates):
            snaps.append(snap)
    return snaps

## Core: metrics and evaluation protocol

Every model is a scorer with `score(snapshot) -> [num_targets, num_candidates]`, so candidate
pools, tie-breaking and metrics are identical across baselines and GNNs. Ties are broken at
random, not by candidate order.

The **primary subset** is test exhibitions that have a recorded curator and at least one returning
artist. Exhibitions whose artists are all MoMA debuts are kept with `n_pos_in_pool = 0` for the
cold-start analysis.

In [ ]:
from sklearn.metrics import average_precision_score, roc_auc_score


def rank_order(scores, rng):
    """Indices sorted by descending score, ties broken uniformly at random."""
    return np.lexsort((rng.random(len(scores)), -scores))


def ranking_metrics(scores, labels, ks, rng):
    """Per-query metrics for one exhibition. `labels` is a boolean vector with >=1 True."""
    order = rank_order(scores, rng)
    rel = labels[order].astype(float)
    n_pos = rel.sum()
    hits = np.cumsum(rel)
    precision_at = hits / np.arange(1, len(rel) + 1)
    discounts = 1.0 / np.log2(np.arange(2, len(rel) + 2))

    out = {}
    for k in ks:
        out[f"recall@{k}"] = hits[min(k, len(rel)) - 1] / n_pos
        out[f"map@{k}"] = (precision_at[:k] * rel[:k]).sum() / min(n_pos, k)
        ideal = discounts[:int(min(n_pos, k))].sum()
        out[f"ndcg@{k}"] = (discounts[:k] * rel[:k]).sum() / ideal
    if n_pos < len(rel):
        out["auc"] = roc_auc_score(labels, scores)
        out["ap"] = average_precision_score(labels, scores)
    else:
        out["auc"] = out["ap"] = np.nan

    ranks = np.empty(len(rel), dtype=np.int64)
    ranks[order] = np.arange(1, len(rel) + 1)
    return out, ranks[labels]


def evaluate(scorer, snap, ks, seed=0):
    """Returns (per-exhibition metrics, per-positive ranks)."""
    rng = np.random.default_rng(seed)
    scores = np.asarray(scorer.score(snap), dtype=np.float64)
    assert scores.shape == (len(snap.targets), len(snap.candidates))
    labels = snap.label_matrix()

    rows, rank_rows = [], []
    for i, e in enumerate(snap.targets):
        n_in_pool = int(labels[i].sum())
        row = {"exh_id": e, "has_curator": bool(snap.has_curator[i]),
               "n_pos": len(snap.positives[e]), "n_pos_in_pool": n_in_pool,
               "n_candidates": len(snap.candidates)}
        if n_in_pool:
            metrics, ranks = ranking_metrics(scores[i], labels[i], ks, rng)
            row.update(metrics)
            rank_rows.append(pd.DataFrame({
                "exh_id": e, "artist": snap.candidates[labels[i]], "rank": ranks,
                "n_candidates": len(snap.candidates)}))
        rows.append(row)
    per_exh = pd.DataFrame(rows)
    ranks = pd.concat(rank_rows, ignore_index=True) if rank_rows else pd.DataFrame()
    return per_exh, ranks


def metric_columns(per_exh):
    return [c for c in per_exh.columns if "@" in c or c in ("auc", "ap")]


def summarize(per_exh):
    """Mean metrics on the curated subset and on all exhibitions with >=1 returning artist."""
    scored = per_exh[per_exh["n_pos_in_pool"] > 0]
    cols = metric_columns(per_exh)
    return {name: {"n_exhibitions": len(subset), **subset[cols].mean().to_dict()}
            for name, subset in [("curated", scored[scored["has_curator"]]), ("all", scored)]}


def paired_bootstrap(a, b, n_boot=1000, seed=0, ci=0.95):
    """CI for mean(a - b), resampling the paired units (test exhibitions)."""
    diff = np.asarray(a, dtype=float) - np.asarray(b, dtype=float)
    diff = diff[~np.isnan(diff)]
    rng = np.random.default_rng(seed)
    means = diff[rng.integers(0, len(diff), size=(n_boot, len(diff)))].mean(axis=1)
    lo, hi = np.quantile(means, [(1 - ci) / 2, 1 - (1 - ci) / 2])
    return {"mean_diff": diff.mean(), "ci_low": lo, "ci_high": hi, "n": len(diff),
            "p_le_0": float((means <= 0).mean())}


def load_per_exh(tag, cutoff):
    """Saved per-exhibition results (averaged over seeds) on the curated subset."""
    files = sorted((RESULTS_DIR / "gnn").glob(f"{tag}_T{cutoff}_s*_per_exh.csv"))
    files = files or sorted((RESULTS_DIR / "baselines").glob(f"{tag}_T{cutoff}_per_exh.csv"))
    if not files:
        return None, 0
    runs = pd.concat([pd.read_csv(f) for f in files])
    scored = runs[(runs["n_pos_in_pool"] > 0) & runs["has_curator"]]
    return scored.groupby("exh_id").mean(numeric_only=True), len(files)


def compare_runs(a, b, metric="map@20", cutoffs=None, n_boot=1000):
    """Paired bootstrap of A - B over curated test exhibitions, per cutoff.

    Tags: baselines by name (e.g. "curator_collab"); GNNs as "{model}_{mode}" (e.g. "rgcn_none").
    """
    rows = []
    for cutoff in cutoffs or CFG["split"]["cutoffs"]:
        (ra, na), (rb, nb) = load_per_exh(a, cutoff), load_per_exh(b, cutoff)
        if ra is None or rb is None:
            continue
        joined = ra[[metric]].merge(rb[[metric]], left_index=True, right_index=True,
                                    suffixes=("_a", "_b"))
        res = paired_bootstrap(joined[f"{metric}_a"], joined[f"{metric}_b"], n_boot=n_boot)
        rows.append({"cutoff": cutoff, "seeds_a": na, "seeds_b": nb,
                     a: joined[f"{metric}_a"].mean(), b: joined[f"{metric}_b"].mean(), **res})
    return pd.DataFrame(rows)

## Load data and check the splits

In [ ]:
tables = load_tables()
data = MomaData.from_tables(tables)
print(f"{data.num_exhibitions} exhibitions, {data.num_artists} artists, "
      f"{data.num_curators} curators, {len(data.artist_edges)} artist edges, "
      f"{len(data.curator_edges)} curator edges")

In [ ]:
# Leakage checks: history strictly precedes targets; training targets end before validation.
begin = data.exhibitions["begin"]
for cutoff in CFG["split"]["cutoffs"]:
    val, test = eval_snapshots(data, cutoff, CFG["split"])
    train = train_snapshots(data, cutoff, CFG["split"])
    for snap in [val, test] + train:
        assert (begin.loc[snap.hist_artist[0]] < snap.history_end).all()
        assert (begin.loc[snap.hist_curator[0]] < snap.history_end).all()
        assert (begin.loc[snap.targets] >= snap.history_end).all()
        assert set(snap.target_curator[0]) <= set(snap.targets)
        assert np.array_equal(snap.candidates, np.unique(snap.hist_artist[1]))
    first_val = begin.loc[val.targets].min()
    assert all((begin.loc[s.targets] < first_val).all() for s in train)
    print(f"T={cutoff}: {len(train)} training snapshots, {len(val.targets)} val / "
          f"{len(test.targets)} test exhibitions, {len(test.candidates)} test candidates")
print("snapshot checks passed")

In [ ]:
# Metric checks on a hand-computed example. Ranking: c0(+), c1(-), c2(+), c3(-), c4(-).
m, r = ranking_metrics(np.array([5, 4, 3, 2, 1.]), np.array([1, 0, 1, 0, 0], bool), [2, 4],
                       np.random.default_rng(0))
assert np.isclose(m["recall@2"], 0.5) and np.isclose(m["recall@4"], 1.0)
assert np.isclose(m["map@2"], 0.5) and np.isclose(m["map@4"], (1 + 2 / 3) / 2)
assert np.isclose(m["ndcg@4"], (1 + 1 / np.log2(4)) / (1 + 1 / np.log2(3)))
assert np.isclose(m["auc"], 5 / 6) and list(r) == [1, 3]
# All-tied scores must not favour the first candidate.
rng = np.random.default_rng(1)
tied = [ranking_metrics(np.zeros(10), np.eye(10, dtype=bool)[0], [1], rng)[1][0]
        for _ in range(200)]
assert 4 < np.mean(tied) < 7
print("metric checks passed")

## Heuristic baselines

In [ ]:
import scipy.sparse as sp
from scipy.stats import rankdata


def incidence(edges, num_nodes, num_cols, weights=None, col_index=None):
    """Sparse [num_nodes, num_cols] matrix from (col, node) edge pairs.

    `col_index` optionally remaps the first edge row (e.g. exh_id -> target position).
    """
    cols = edges[0] if col_index is None else col_index[edges[0]]
    w = np.ones(edges.shape[1]) if weights is None else weights
    return sp.csr_matrix((w, (edges[1], cols)), shape=(num_nodes, num_cols))


def target_index(snap, num_exhibitions):
    pos = np.full(num_exhibitions, -1)
    pos[snap.targets] = np.arange(len(snap.targets))
    return pos


def lexicographic(primary, secondary):
    """Order by `primary`, then `secondary` (a per-candidate vector)."""
    ranks = rankdata(primary, method="dense", axis=1)
    return ranks * (secondary.max() + 1) + secondary[None, :]


def artist_popularity(snap, num_artists):
    return np.bincount(snap.hist_artist[1], minlength=num_artists).astype(float)


class Random:
    name = "random"

    def score(self, snap):
        return np.zeros((len(snap.targets), len(snap.candidates)))


class Popularity:
    name = "popularity"

    def __init__(self, data):
        self.data = data

    def score(self, snap):
        pop = artist_popularity(snap, self.data.num_artists)[snap.candidates]
        return np.tile(pop, (len(snap.targets), 1))


class RecencyPopularity:
    name = "recency_popularity"

    def __init__(self, data, tau):
        self.data, self.tau = data, tau

    def score(self, snap):
        years = self.data.exhibitions["year"].to_numpy()[snap.hist_artist[0]]
        w = np.exp(-(snap.ref_year - years) / self.tau)
        pop = np.bincount(snap.hist_artist[1], weights=w, minlength=self.data.num_artists)
        return np.tile(pop[snap.candidates], (len(snap.targets), 1))


class CuratorCollab:
    name = "curator_collab"

    def __init__(self, data):
        self.data = data

    def score(self, snap):
        d = self.data
        n_exh = d.num_exhibitions
        artist_exh = incidence(snap.hist_artist, d.num_artists, n_exh)
        curator_exh = incidence(snap.hist_curator, d.num_curators, n_exh)
        shared = artist_exh @ curator_exh.T                          # artists x curators
        target_cur = incidence(snap.target_curator, d.num_curators, len(snap.targets),
                               col_index=target_index(snap, n_exh))  # curators x targets
        primary = (shared @ target_cur).toarray()[snap.candidates].T
        pop = artist_popularity(snap, d.num_artists)[snap.candidates]
        return lexicographic(primary, pop)


class PersonGraph:
    """Projected person-person co-exhibition graph over history exhibitions. Artists and
    curators are merged by constituent id, so a person with both roles is one node."""

    def __init__(self, data):
        cids = pd.Index(np.union1d(data.artists["cid"], data.curators["cid"]))
        self.num_persons = len(cids)
        self.artist_person = cids.get_indexer(data.artists["cid"])
        self.curator_person = cids.get_indexer(data.curators["cid"])
        self.data = data

    def adjacency(self, snap):
        n_exh = self.data.num_exhibitions
        edges = np.concatenate([
            np.stack([snap.hist_artist[0], self.artist_person[snap.hist_artist[1]]]),
            np.stack([snap.hist_curator[0], self.curator_person[snap.hist_curator[1]]]),
        ], axis=1)
        person_exh = incidence(edges, self.num_persons, n_exh)
        person_exh.data[:] = 1.0
        adj = (person_exh @ person_exh.T).tocsr()
        adj.setdiag(0)
        adj.eliminate_zeros()
        adj.sort_indices()
        adj.data[:] = 1.0
        return adj

    def target_curators(self, snap):
        """Sparse [persons, targets] indicator of each target's curators."""
        n_exh = self.data.num_exhibitions
        edges = np.stack([snap.target_curator[0], self.curator_person[snap.target_curator[1]]])
        m = incidence(edges, self.num_persons, len(snap.targets),
                      col_index=target_index(snap, n_exh))
        m.data[:] = 1.0
        return m


class CommonNeighbors:
    """Sum over the target's curators c of |N(a) ∩ N(c)| in the person graph."""
    name = "common_neighbors"
    adamic_adar = False

    def __init__(self, data):
        self.graph = PersonGraph(data)
        self.data = data

    def score(self, snap):
        adj = self.graph.adjacency(snap)
        reach = adj @ self.graph.target_curators(snap)               # persons x targets
        if self.adamic_adar:
            deg = np.asarray(adj.sum(axis=1)).ravel()
            inv_log = np.where(deg > 1, 1.0 / np.log(np.maximum(deg, 2)), 0.0)
            reach = sp.diags(inv_log) @ reach
        primary = (adj @ reach).toarray()[self.graph.artist_person[snap.candidates]].T
        pop = artist_popularity(snap, self.data.num_artists)[snap.candidates]
        return lexicographic(primary, pop)


class AdamicAdar(CommonNeighbors):
    """Like common neighbors, but each shared neighbor z counts 1 / log(deg(z))."""
    name = "adamic_adar"
    adamic_adar = True

## Node2Vec

This implementation has no `torch_cluster` or `pyg-lib` dependency, so it runs anywhere. Biased
second-order walks use vectorized rejection sampling, and embeddings are trained with skip-gram
plus negative sampling.

In [ ]:
import torch
import torch.nn.functional as F


def random_walks(adj, starts, length, p, q, rng):
    indptr, indices = adj.indptr, adj.indices
    n = adj.shape[0]
    deg = np.diff(indptr)
    edge_keys = np.repeat(np.arange(n, dtype=np.int64), deg) * n + indices  # sorted (CSR)

    def uniform_neighbor(nodes):
        return indices[indptr[nodes] + (rng.random(len(nodes)) * deg[nodes]).astype(np.int64)]

    def is_edge(u, v):
        keys = u.astype(np.int64) * n + v
        pos = np.minimum(np.searchsorted(edge_keys, keys), len(edge_keys) - 1)
        return edge_keys[pos] == keys

    walks = np.empty((len(starts), length), dtype=np.int64)
    walks[:, 0] = starts
    walks[:, 1] = uniform_neighbor(starts)
    max_w = max(1 / p, 1.0, 1 / q)
    for step in range(2, length):
        prev, cur = walks[:, step - 2], walks[:, step - 1]
        if p == 1 and q == 1:
            walks[:, step] = uniform_neighbor(cur)
            continue
        pending = np.arange(len(cur))
        while len(pending):
            cand = uniform_neighbor(cur[pending])
            pr = prev[pending]
            w = np.where(cand == pr, 1 / p, np.where(is_edge(pr, cand), 1.0, 1 / q))
            accept = rng.random(len(pending)) < w / max_w
            walks[pending[accept], step] = cand[accept]
            pending = pending[~accept]
    return walks


def skipgram_pairs(walks, window):
    pairs = []
    for offset in range(1, window + 1):
        a, b = walks[:, :-offset].ravel(), walks[:, offset:].ravel()
        pairs += [np.stack([a, b]), np.stack([b, a])]
    return np.concatenate(pairs, axis=1)


def train_embeddings(adj, cfg, seed):
    rng = np.random.default_rng(seed)
    torch.manual_seed(seed)
    deg = np.diff(adj.indptr)
    starts = np.tile(np.flatnonzero(deg > 0), cfg["walks_per_node"])
    walks = random_walks(adj, starts, cfg["walk_length"], cfg["p"], cfg["q"], rng)
    pairs = torch.from_numpy(skipgram_pairs(walks, cfg["window"]))

    n, dim = adj.shape[0], cfg["dim"]
    emb_in = torch.nn.Embedding(n, dim)
    emb_out = torch.nn.Embedding(n, dim)
    torch.nn.init.normal_(emb_in.weight, std=0.1)
    torch.nn.init.zeros_(emb_out.weight)
    noise = torch.from_numpy(deg.astype(np.float64) ** 0.75)
    opt = torch.optim.Adam(list(emb_in.parameters()) + list(emb_out.parameters()), lr=0.01)

    batch = 8192
    for _ in range(cfg["epochs"]):
        perm = torch.randperm(pairs.shape[1])
        for i in range(0, len(perm), batch):
            u, v = pairs[:, perm[i:i + batch]]
            neg = torch.multinomial(noise, len(u) * cfg["num_neg"], replacement=True)
            zu = emb_in(u)
            pos = F.logsigmoid((zu * emb_out(v)).sum(-1))
            negs = F.logsigmoid(-(zu.unsqueeze(1) * emb_out(neg).view(len(u), cfg["num_neg"], dim))
                                .sum(-1)).sum(-1)
            loss = -(pos + negs).mean()
            opt.zero_grad()
            loss.backward()
            opt.step()
    return F.normalize(emb_in.weight.detach(), dim=-1).numpy(), deg > 0


class Node2VecScorer:
    """Score(a, e) = cos(z_a, mean of z_c over e's curators with history)."""
    name = "node2vec"

    def __init__(self, data, cfg, seed=0):
        self.graph = PersonGraph(data)
        self.data, self.cfg, self.seed = data, cfg, seed

    def score(self, snap):
        adj = self.graph.adjacency(snap)
        z, has_history = train_embeddings(adj, self.cfg, self.seed)
        cur = self.graph.target_curators(snap).multiply(has_history[:, None]).tocsc()
        counts = np.asarray(cur.sum(axis=0)).ravel()
        query = (cur.T @ z) / np.maximum(counts, 1)[:, None]          # targets x dim
        primary = query @ z[self.graph.artist_person[snap.candidates]].T
        pop = artist_popularity(snap, self.data.num_artists)[snap.candidates]
        return lexicographic(primary, pop)

## Run all baselines

In [ ]:
RUN_NODE2VEC = True     # about 40 s per cutoff on a laptop CPU
CUTOFFS = CFG["split"]["cutoffs"]

out_dir = RESULTS_DIR / "baselines"
out_dir.mkdir(exist_ok=True)
scorers = [Random(), Popularity(data), RecencyPopularity(data, CFG["baselines"]["recency_tau"]),
           CuratorCollab(data), CommonNeighbors(data), AdamicAdar(data)]
if RUN_NODE2VEC:
    scorers.append(Node2VecScorer(data, CFG["baselines"]["node2vec"]))

rows = []
for cutoff in CUTOFFS:
    _, test = eval_snapshots(data, cutoff, CFG["split"])
    for scorer in scorers:
        start = time.time()
        per_exh, ranks = evaluate(scorer, test, CFG["eval"]["ks"])
        per_exh.to_csv(out_dir / f"{scorer.name}_T{cutoff}_per_exh.csv", index=False)
        ranks.to_csv(out_dir / f"{scorer.name}_T{cutoff}_ranks.csv", index=False)
        for subset, metrics in summarize(per_exh).items():
            rows.append({"model": scorer.name, "cutoff": cutoff, "subset": subset, **metrics})
        print(f"T={cutoff} {scorer.name:20s} {time.time() - start:5.1f}s")

baseline_table = pd.DataFrame(rows)
baseline_table.to_csv(out_dir / "summary.csv", index=False)

In [ ]:
cols = ["recall@10", "recall@50", "map@20", "ndcg@20", "auc"]
curated = baseline_table[baseline_table["subset"] == "curated"]
curated.pivot_table(index="model", columns="cutoff", values="map@20").round(4)

In [ ]:
# Mean over cutoffs, curated subset.
curated.groupby("model")[cols].mean().sort_values("map@20", ascending=False).round(4)

## Is there curator signal at all?

This is a paired bootstrap over test exhibitions of curator_collab − popularity. A CI above
zero means that knowing the curator improves ranking over popularity alone.

In [ ]:
compare_runs("curator_collab", "popularity", metric="map@20").round(4)